In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
!pip install -U transformers accelerate peft bitsandbytes datasets 
!pip install flash-attn --no-build-isolation
import torch
torch.set_float32_matmul_precision('high') # Enable TF32 for A100

In [ ]:
# 1. Clear previous failed attempts and install dependencies
print(">>> [ENV] Installing Enterprise-grade dependencies...")
!pip install -U transformers accelerate peft bitsandbytes datasets einops

# 2. Fix for Flash Attention in Cloud Environments
# Using --no-cache-dir avoids the 'Invalid cross-device link' error
print(">>> [ENV] Building Flash Attention 2 (Hardware-Native)...")
!pip install flash-attn --no-build-isolation --no-cache-dir

# 3. Hardware Verification
import torch
print(f">>> [CHECK] CUDA Version: {torch.version.cuda}")
print(f">>> [CHECK] GPU: {torch.cuda.get_device_name(0)}")
torch.set_float32_matmul_precision('high')

In [ ]:
import os
import logging
import torch
import torch.nn as nn
from dataclasses import dataclass, field
from typing import Optional, Dict, List, Any, Tuple
from PIL import Image

# Third-party Imports
from datasets import load_dataset
from transformers import (
    PreTrainedModel, PretrainedConfig, Qwen2TokenizerFast, 
    CLIPImageProcessor, CLIPVisionModel, Qwen2ForCausalLM,
    TrainingArguments, Trainer
)
from peft import LoraConfig, get_peft_model, TaskType

# ------------------------------------------------------------------------------
# 1. LOGGING SETUP
# ------------------------------------------------------------------------------
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# ------------------------------------------------------------------------------
# 2. CONFIGURATION MANAGEMENT
# ------------------------------------------------------------------------------
@dataclass
class ModelArguments:
    """Arguments handling model architecture and paths."""
    llm_model_id: str = "Qwen/Qwen2.5-3B-Instruct"
    vision_model_id: str = "openai/clip-vit-large-patch14"
    projector_path: Optional[str] = "projector_v3.pt"
    image_token_count: int = 256

@dataclass
class DataArguments:
    """Arguments handling dataset and processing."""
    dataset_name: str = "HuggingFaceH4/llava-instruct-mix-vsft"
    max_seq_length: int = 2048

# ------------------------------------------------------------------------------
# 3. MODEL DEFINITION
# ------------------------------------------------------------------------------
class VLMConfig(PretrainedConfig):
    model_type = "enterprise_vlm"
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

class Projector(nn.Module):
    def __init__(self, in_dim: int, out_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, out_dim),
            nn.GELU(),
            nn.Linear(out_dim, out_dim)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

class EnterpriseVLM(PreTrainedModel):
    config_class = VLMConfig
    
    def __init__(self, config: VLMConfig, model_args: ModelArguments):
        super().__init__(config)
        self.image_token_count = model_args.image_token_count
        
        # Load Sub-Components
        self.vision_model = CLIPVisionModel.from_pretrained(model_args.vision_model_id)
        self.llm = Qwen2ForCausalLM.from_pretrained(
            model_args.llm_model_id,
            torch_dtype=torch.bfloat16,
            attn_implementation="flash_attention_2"
        )
        self.projector = Projector(
            self.vision_model.config.hidden_size, 
            self.llm.config.hidden_size
        )

    def gradient_checkpointing_enable(self, gradient_checkpointing_kwargs=None):
        """Forward gradient checkpointing to the LLM backend."""
        self.llm.gradient_checkpointing_enable(gradient_checkpointing_kwargs=gradient_checkpointing_kwargs)

    def forward(
        self, 
        input_ids: torch.LongTensor, 
        labels: Optional[torch.LongTensor] = None, 
        pixel_values: Optional[torch.FloatTensor] = None,
        **kwargs
    ):
        # 1. Vision Forward (Frozen)
        with torch.no_grad():
            vis_features = self.vision_model(pixel_values).last_hidden_state[:, 1:]

        # 2. Projection (Trainable)
        vis_features = self.projector(vis_features.to(dtype=self.llm.dtype))

        # 3. Pooling (If necessary)
        if vis_features.shape[1] != self.image_token_count:
            vis_features = torch.nn.functional.adaptive_avg_pool1d(
                vis_features.transpose(1, 2), self.image_token_count
            ).transpose(1, 2)

        # 4. Text Forward
        txt_embeds = self.llm.get_input_embeddings()(input_ids)

        # 5. Multimodal Fusion
        # Concatenate [Image_Embeds] + [Text_Embeds excluding placeholders]
        combined_embeds = torch.cat(
            (vis_features, txt_embeds[:, self.image_token_count:]), 
            dim=1
        )

        return self.llm(inputs_embeds=combined_embeds, labels=labels)

# ------------------------------------------------------------------------------
# 4. DATA COLLATOR (Enterprise Standard)
# ------------------------------------------------------------------------------
@dataclass
class VLMCollator:
    """
    Standard DataCollator pattern. 
    Processes a list of raw samples into a batch tensor.
    """
    tokenizer: Any
    image_processor: Any
    image_token_count: int
    max_length: int

    def __call__(self, batch: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        # Filter corrupted samples
        batch = [x for x in batch if x is not None]
        if not batch: return {}

        # 1. Process Images
        images = []
        for item in batch:
            if item.get("image") is not None:
                images.append(item["image"])
            else:
                # Black Image Fallback (Text-Only Support)
                images.append(Image.new('RGB', (224, 224), color=(0, 0, 0)))
        
        pixel_values = self.image_processor(images, return_tensors="pt").pixel_values

        # 2. Process Text
        texts = [item["text"] for item in batch]
        
        # Dynamic Padding (Standard HF Practice)
        tokenized = self.tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=self.max_length - self.image_token_count,
            return_tensors="pt",
            add_special_tokens=False
        )

        # 3. Construct Inputs (Placeholders + Text)
        batch_size = tokenized.input_ids.shape[0]
        pad_tokens = torch.full((batch_size, self.image_token_count), self.tokenizer.pad_token_id, dtype=torch.long)
        
        full_input_ids = torch.cat([pad_tokens, tokenized.input_ids], dim=1)
        
        # 4. Construct Labels (Masking)
        labels = full_input_ids.clone()
        labels[:, :self.image_token_count] = -100  # Mask Image Region
        labels[labels == self.tokenizer.pad_token_id] = -100 # Mask Padding

        return {
            "input_ids": full_input_ids,
            "labels": labels,
            "pixel_values": pixel_values
        }

# ------------------------------------------------------------------------------
# 5. DATASET PARSING
# ------------------------------------------------------------------------------
def parse_llava_data(example: Dict) -> Optional[Dict]:
    """Helper to parse raw LLaVA dataset samples."""
    try:
        # 1. Extract Image
        image = None
        if "images" in example and example["images"] and len(example["images"]) > 0:
            image = example["images"][0].convert("RGB")
        
        # 2. Extract Text
        full_text = ""
        for msg in example["messages"]:
            role = "user" if msg["role"] == "user" else "assistant"
            full_text += f"<|im_start|>{role}\n"
            
            content = msg["content"]
            if isinstance(content, str):
                full_text += content
            elif isinstance(content, list):
                for part in content:
                    if part.get("type") == "text":
                        full_text += part["text"]
            
            full_text += "<|im_end|>\n"
        full_text += "<|endoftext|>"

        return {"image": image, "text": full_text}
    except Exception as e:
        logger.warning(f"Skipping corrupt sample: {e}")
        return None

# ------------------------------------------------------------------------------
# 6. MAIN EXECUTION
# ------------------------------------------------------------------------------
def main():
    # 1. Setup Arguments
    model_args = ModelArguments()
    data_args = DataArguments()
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    logger.info(f"Starting VLM Training on {device}")

    # 2. Load Artifacts
    tokenizer = Qwen2TokenizerFast.from_pretrained(model_args.llm_model_id)
    # Ensure padding token exists
    if tokenizer.pad_token is None: 
        tokenizer.pad_token_id = 151643 
        
    image_processor = CLIPImageProcessor.from_pretrained(model_args.vision_model_id)

    # 3. Prepare Dataset
    logger.info(f"Loading Dataset: {data_args.dataset_name}")
    raw_dataset = load_dataset(data_args.dataset_name, split="train")
    
    # Pre-process dataset (Lightweight mapping)
    train_dataset = raw_dataset.map(
        parse_llava_data,
        remove_columns=raw_dataset.column_names,
        num_proc=os.cpu_count(),
        desc="Parsing Dataset"
    ).filter(
        lambda x: x is not None, 
        num_proc=os.cpu_count(), # <--- ADD THIS LINE (Uses all cores)
        desc="Filtering Nones"   # <--- Optional: Adds a label to the bar
    )

    # 4. Initialize Model
    logger.info("Initializing EnterpriseVLM...")
    config = VLMConfig()
    model = EnterpriseVLM(config, model_args)

    # Load Stage 1 Projector
    if os.path.exists(model_args.projector_path):
        logger.info(f"Loading Projector weights from {model_args.projector_path}")
        model.projector.load_state_dict(torch.load(model_args.projector_path))
    else:
        logger.warning("Projector weights not found! Initializing from scratch.")

    # 5. Apply LoRA (PEFT)
    logger.info("Applying LoRA Configuration...")
    lora_config = LoraConfig(
        r=64,
        lora_alpha=128,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        task_type=TaskType.CAUSAL_LM,
        bias="none"
    )
    model.llm = get_peft_model(model.llm, lora_config)

    # 6. Freeze / Unfreeze
    model.vision_model.requires_grad_(False)
    model.projector.requires_grad_(True)
    model.to(device)
    
    model.llm.print_trainable_parameters()

    # 7. Training Arguments
    training_args = TrainingArguments(
        output_dir="./vlm_enterprise_checkpoints",
        per_device_train_batch_size=4,
        gradient_accumulation_steps=16,      # Effective Batch Size = 128
        learning_rate=5e-5,
        warmup_ratio=0.1,
        num_train_epochs=1,
        bf16=True,                          # A100 Requirement
        logging_steps=1,
        save_strategy="steps",
        save_steps=100,
        save_total_limit=2,
        gradient_checkpointing=True,        # Memory Optimization
        optim="adamw_torch",
        report_to="none",
        dataloader_num_workers=16,
        dataloader_pin_memory=True,
        remove_unused_columns=False         # Required for custom Collator
    )

    # 8. Initialize Trainer
    collator = VLMCollator(
        tokenizer=tokenizer,
        image_processor=image_processor,
        image_token_count=model_args.image_token_count,
        max_length=data_args.max_seq_length
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        data_collator=collator
    )

    # 9. Train
    logger.info("Starting Training Loop...")
    trainer.train()

    # 10. Save Artifacts
    logger.info("Saving Final Model...")
    output_path = "./final_vlm_model"
    model.llm.save_pretrained(f"{output_path}/adapter")
    torch.save(model.projector.state_dict(), f"{output_path}/projector.pt")
    tokenizer.save_pretrained(output_path)

if __name__ == "__main__":
    main()